# logit-thermo A2-12：1 回の禁止で乱れるトークンの数 k（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 20〜30 分。終わったら自動で切れます。
- ゲート：`docs/A2_12_GATE.md`。同じ途中の状態から「何もしない」と「1 位を 1 回だけ禁止」の 2 本を、同じノイズで 64 歩生成して、食い違うトークンの数を数え、
  凝固点降下のずれから先に決めた予想の k と比べて `A2_12/a2_12_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・A2-7 のデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_12_kdist.py`（`08363b2061241d94…`）は埋め込み（分析の部分はデータを開く前に固定）。
- 保存先 `A2_12/`。途中で切れても、もう一度「すべて実行」すれば終わったモデルは飛ばします。
- **通知**：🚀・温度ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-12 1 回の禁止で乱れるトークンの数 k（3 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-12 1 回の禁止で乱れるトークンの数 k（3 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-12 tokens disturbed per ban', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import gc, math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550'})
SRC22, SRC7C = _s['SRC22'], _s['SRC7C']
SRC12 = '"""A2-12（ゲート docs/A2_12_GATE.md）：1 回の禁止で乱れるトークンの数 k を測り、凝固点降下と熱量計の ΔH のずれから先に決めた k の予想と比べる。\nGPU（L4）で生成、そのあと同じノートブックで分析（CPU の計算）。\nscripts/a2_2_collect.py（load_model・valid_mask・split_doc・load_docs・SEED）と scripts/a2_7_collect.py（CopyIndexN）を実行したのと同じ名前空間で実行する。\n分析の部分はデータを開く前に固定する。\n\nやり方：\n- 書き出し（A2-2 と同じ 40 本の記事の先頭 64 トークン）から、温度 T でガンベルの最大（ノイズ g を足して一番大きいトークンを選ぶ。\n  ふつうのサンプリングと同じ分布）で 256 歩の「下書き」を作る。\n- 分かれ目 t0 ＝ 64・128・192・256 のそれぞれで、下書きの t0 歩目までを共通にして 2 本に分ける。\n  a：何もしない。b：t0 歩目だけ、1 位のトークンを禁止する（A2-10 の偽薬と同じく、1 位が 3 トークンの写しの候補でないときだけ。\n  候補なら、その分かれ目は数えない）。\n- 分かれ目から 64 歩、2 本で同じノイズ g（歩ごと・記事ごと）を使う（共通の乱数）。分布がほとんど同じならトークンも同じになる。\n- k ＝ その 64 歩で 2 本のトークンが食い違った歩の数（分かれ目の歩を含む）。a が分かれ目で 1 位を選ばなかったときは、b も同じトークンを\n  選ぶので、その後もずっと同じ（k ＝ 0）。これも A2-10 の偽薬の 1 回の禁止と同じ（濃さは禁止の回数で数えている）。\n- 記録：歩ごとの 2 本の分布の全変動距離（TV、0〜1）とその和、歩ごとの食い違いの割合（曲線）、分かれ目での 1 位の確率。\n"""\nimport gc, math, os, time\nimport numpy as np\nimport torch\n\nK12_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')]\nK12_TEMPS = [0.5, 0.55, 0.6]\nK12_T_MAIN = 0.55\nK12_FORKS = [64, 128, 192, 256]\nK12_W = 64\nK12_PREFIX = 256\nK12_SEED = 20261119\n# 予想の k（A2-10 §4 と A2-8 の数から、データを見る前に決まっている）：k_pred ＝ 熱量計の ΔH ÷ 凝固点降下の ΔH（x ＝ 濃さ）\nK12_DH_CAL = {\'pythia-1b\': 1.2785, \'olmo2-1b\': 1.3645, \'qwen2.5-1.5b\': 1.4053}      # A2-8（ナット／トークン）\nK12_DH_COL = {\'pythia-1b\': 1.5301, \'olmo2-1b\': 0.764344, \'qwen2.5-1.5b\': 0.721599}  # A2-10（ナット、x ＝ 濃さ）\nK12_PRED = {m: K12_DH_CAL[m] / K12_DH_COL[m] for m in K12_DH_CAL}\nK12_BAND = 1.5\nK12_BOOT, K12_BOOT_SEED = 1000, 20261120\nK12_Q2_BINS = [(1.0, \'1 未満\'), (2.0, \'1〜2\'), (4.0, \'2〜4\'), (float(\'inf\'), \'4 以上\')]\n\n\ndef prefill(model, x):\n    """最後の位置の logit だけ作る（全部の位置の logit を作ると、Qwen では数十 GB になる）。"""\n    try:\n        return model(x, use_cache=True, logits_to_keep=1)\n    except TypeError:\n        return model(x, use_cache=True)\n\n\ndef gumbel(shape, seed, device):\n    g = torch.Generator(device=device).manual_seed(int(seed))\n    u = torch.rand(shape, generator=g, device=device, dtype=torch.float64).clamp_(1e-12, 1 - 1e-12)\n    return -torch.log(-torch.log(u))\n\n\ndef choose(z, T, valid, g):\n    """z：(B, V) の logit。→ ガンベルの最大で選んだトークン (B,)。z/T ＋ g の最大 ＝ softmax(z/T) からのサンプル。"""\n    s = (z.double() / T).masked_fill(~valid, float(\'-inf\')) + g\n    return s.argmax(-1)\n\n\n@torch.no_grad()\ndef make_prefix(model, prompts, T, valid, seed):\n    """→ (B, K12_PREFIX) の下書き（何も禁止しない）。"""\n    device = valid.device\n    x = torch.tensor(prompts, device=device)\n    out = prefill(model, x)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    toks = []\n    for t in range(K12_PREFIX):\n        nxt = choose(logits, T, valid, gumbel(logits.shape, seed + t, device))\n        toks.append(nxt)\n        out = model(nxt[:, None], past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    return torch.stack(toks, 1).cpu().numpy()\n\n\n@torch.no_grad()\ndef fork_once(model, prompts, prefix, t0, T, valid, seed, ban=True):\n    """t0 で分ける。→ dict(elig (B,), diff (B, W), tv (B, W), p1 (B,), chose1 (B,))。ban＝False なら b も禁止しない（自己テスト用）。"""\n    device = valid.device\n    B = len(prompts)\n    hist = [list(prompts[i]) + list(map(int, prefix[i, :t0])) for i in range(B)]\n    x = torch.tensor(hist + hist, device=device)                        # 2B 本：前半が a、後半が b\n    out = prefill(model, x)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    z0 = logits[:B].double().masked_fill(~valid, float(\'-inf\'))\n    top1 = z0.argmax(-1)\n    cand = [CopyIndexN(h, 3).candidates() for h in hist]\n    elig = np.array([int(top1[i]) not in set(cand[i]) for i in range(B)])\n    p1 = torch.softmax(z0 / T, -1).gather(-1, top1[:, None])[:, 0].cpu().numpy()\n    diff = np.zeros((B, K12_W), np.int8); tv = np.zeros((B, K12_W), np.float32); chose1 = np.zeros(B, bool)\n    for t in range(K12_W):\n        za = logits[:B].double().masked_fill(~valid, float(\'-inf\'))\n        zb = logits[B:].double().masked_fill(~valid, float(\'-inf\'))\n        if t == 0 and ban:\n            rows = torch.tensor(np.flatnonzero(elig), device=device, dtype=torch.long)\n            if len(rows):\n                zb[rows, top1[rows]] = float(\'-inf\')\n        pa, pb = torch.softmax(za / T, -1), torch.softmax(zb / T, -1)\n        tv[:, t] = (0.5 * (pa - pb).abs().sum(-1)).float().cpu().numpy()\n        g = gumbel((B, za.shape[-1]), seed + t, device)\n        ta = (za / T + g).argmax(-1); tb = (zb / T + g).argmax(-1)\n        if t == 0:\n            chose1 = (ta == top1).cpu().numpy()\n        diff[:, t] = (ta != tb).cpu().numpy()\n        nxt = torch.cat([ta, tb])\n        out = model(nxt[:, None], past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    return dict(elig=elig, diff=diff, tv=tv, p1=p1, chose1=chose1)\n\n\ndef run_k12(name, hf_id, docs, outdir, device, notify=print):\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    t0_ = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, _ = valid_mask(tok, model, device)\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    prompts = [pairs[i][0] for i in use]\n    assert len({len(p) for p in prompts}) == 1\n    nT, nF, B = len(K12_TEMPS), len(K12_FORKS), len(prompts)\n    save = {k: np.zeros((nT, nF, B) + s, dt) for k, s, dt in [(\'elig\', (), bool), (\'chose1\', (), bool), (\'p1\', (), np.float32),\n                                                               (\'diff\', (K12_W,), np.int8), (\'tv\', (K12_W,), np.float32)]}\n    prefixes = np.zeros((nT, B, K12_PREFIX), np.int32)\n    for k, T in enumerate(K12_TEMPS):\n        pre = make_prefix(model, prompts, T, valid, K12_SEED + 100000 * k)\n        prefixes[k] = pre\n        for f, t0 in enumerate(K12_FORKS):\n            r = fork_once(model, prompts, pre, t0, T, valid, K12_SEED + 100000 * k + 10000 * (f + 1))\n            for kk in save:\n                save[kk][k, f] = r[kk]\n        notify(f\'{name} 温度 {T} おわり（{time.time() - t0_:.0f} 秒）\')\n    np.savez_compressed(path + \'.part.npz\', prefixes=prefixes, use=np.array(use), temps=np.array(K12_TEMPS), forks=np.array(K12_FORKS), **save)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef self_test_k12(device):\n    """小さなランダムのモデルで：(1) 禁止しなければ 2 本はずっと同じ。(2) a が分かれ目で 1 位を選ばなかった行は、禁止しても 2 本はずっと同じ。\n    (3) a が 1 位を選んだ禁止できる行は、分かれ目で必ず食い違う。(4) ガンベルの最大の選び方の頻度が softmax に近い。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global K12_W, K12_PREFIX\n    keep = (K12_W, K12_PREFIX)\n    K12_W, K12_PREFIX = 12, 20\n    try:\n        prompts = [[5 + i, 6, 7, 8] * 4 for i in range(16)]\n        pre = make_prefix(model, prompts, 0.3, valid, 1)\n        r0 = fork_once(model, prompts, pre, 10, 0.3, valid, 2, ban=False)\n        assert r0[\'diff\'].sum() == 0 and np.abs(r0[\'tv\']).max() < 1e-4\n        r = fork_once(model, prompts, pre, 10, 0.3, valid, 2, ban=True)\n        same = ~(r[\'elig\'] & r[\'chose1\'])\n        assert r[\'diff\'][same].sum() == 0\n        r = fork_once(model, prompts, pre, 10, 0.001, valid, 2, ban=True)        # ほぼ貪欲：a は必ず 1 位を選ぶ\n        same = ~(r[\'elig\'] & r[\'chose1\'])\n        assert r[\'diff\'][same].sum() == 0\n        hit = r[\'elig\'] & r[\'chose1\']\n        assert hit.any() and (r[\'diff\'][hit, 0] == 1).all()\n        # ガンベルの最大：小さな分布で頻度を確かめる\n        z = torch.tensor([[2.0, 1.0, 0.0, -1.0]], device=device).repeat(20000, 1)\n        v = torch.ones(4, dtype=torch.bool, device=device)\n        c = choose(z, 1.0, v, gumbel(z.shape, 3, device)).cpu().numpy()\n        p = torch.softmax(z[0].double(), -1).cpu().numpy()\n        assert np.abs(np.bincount(c, minlength=4) / len(c) - p).max() < 0.015\n    finally:\n        K12_W, K12_PREFIX = keep\n    return \'OK\'\n\n\n# ---------- 分析（CPU） ----------\n\ndef k_stats(Z, T, boot_seed=K12_BOOT_SEED):\n    """→ 温度 T での k（禁止できた分かれ目の食い違いの数の平均）と、記事を選び直した 95% 区間。"""\n    k = list(Z[\'temps\']).index(T)\n    elig = Z[\'elig\'][k]                                   # (nF, B)\n    dsum = Z[\'diff\'][k].astype(np.float64).sum(-1)        # (nF, B)\n    tvs = Z[\'tv\'][k].astype(np.float64).sum(-1)\n    B = elig.shape[1]\n    num = (dsum * elig).sum(0); den = elig.sum(0).astype(float)\n    numt = (tvs * elig).sum(0)\n    kp = float(num.sum() / den.sum()) if den.sum() else float(\'nan\')\n    rng = np.random.default_rng(boot_seed)\n    bs = []\n    for _ in range(K12_BOOT):\n        ii = rng.integers(0, B, B)\n        if den[ii].sum():\n            bs.append(num[ii].sum() / den[ii].sum())\n    ci = [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))]\n    curve = ((Z[\'diff\'][k].astype(np.float64) * elig[..., None]).sum((0, 1)) / den.sum()).tolist()\n    return dict(k=kp, ci=ci, k_tv=float(numt.sum() / den.sum()), n_forks=int(den.sum()), elig_frac=float(elig.mean()),\n                chose1_frac=float((Z[\'chose1\'][k] & elig).sum() / den.sum()), p1_mean=float(Z[\'p1\'][k][elig].mean()),\n                curve=curve, late16=float(np.mean(curve[-16:])))\n\n\ndef analyze12(Zs):\n    res = {\'models\': {}, \'pred\': K12_PRED}\n    for name, _ in K12_MODELS:\n        Z = Zs[name]\n        row = {str(T): k_stats(Z, T) for T in K12_TEMPS}\n        m = row[str(K12_T_MAIN)]\n        pr = K12_PRED[name]\n        m[\'within\'] = bool(pr / K12_BAND <= m[\'k\'] <= pr * K12_BAND)\n        res[\'models\'][name] = row\n    res[\'KQ1\'] = sum(res[\'models\'][n][str(K12_T_MAIN)][\'within\'] for n, _ in K12_MODELS)\n    med = float(np.median([res[\'models\'][n][str(K12_T_MAIN)][\'k\'] for n, _ in K12_MODELS]))\n    res[\'k_median\'] = med\n    res[\'KQ2\'] = next(w for e, w in K12_Q2_BINS if med < e)\n    return res\n\n\ndef self_test_a12():\n    """作り物：分かれ目で 1 位を選ぶ確率 0.6、選んだら平均 3 歩食い違う → k ≈ 1.8。予想の帯の判定。"""\n    rng = np.random.default_rng(0)\n    nT, nF, B = len(K12_TEMPS), len(K12_FORKS), 40\n    Zs = {}\n    for name, _ in K12_MODELS:\n        elig = rng.random((nT, nF, B)) < 0.8\n        chose = rng.random((nT, nF, B)) < 0.6\n        diff = np.zeros((nT, nF, B, K12_W), np.int8)\n        L = rng.geometric(1 / 3, (nT, nF, B))\n        for idx in np.ndindex(nT, nF, B):\n            if elig[idx] and chose[idx]:\n                diff[idx][:min(L[idx], K12_W)] = 1\n        Zs[name] = dict(temps=np.array(K12_TEMPS), elig=elig, chose1=chose & elig, p1=np.full((nT, nF, B), 0.6, np.float32),\n                        diff=diff, tv=diff.astype(np.float32) * 0.5)\n    r = analyze12(Zs)\n    k = r[\'models\'][\'pythia-1b\'][str(K12_T_MAIN)][\'k\']\n    assert 1.4 < k < 2.2, k\n    assert r[\'KQ1\'] == 2 and r[\'KQ2\'] == \'1〜2\', (r[\'KQ1\'], r[\'KQ2\'], {n: r[\'models\'][n][str(K12_T_MAIN)][\'k\'] for n, _ in K12_MODELS})\n    assert abs(K12_PRED[\'pythia-1b\'] - 0.8356) < 1e-3 and abs(K12_PRED[\'olmo2-1b\'] - 1.7852) < 1e-3 and abs(K12_PRED[\'qwen2.5-1.5b\'] - 1.9475) < 1e-3\n    return \'OK\'\n'
assert hashlib.sha256(SRC12.encode()).hexdigest() == '08363b2061241d94875dbf07496c6e98b991465993a768c52bf75c6691f2305a'
exec(SRC22)
exec(SRC7C)
exec(SRC12)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 生成（モデルごと）→ 分析 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a12(), self_test_k12('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_12', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    for name, hf_id in K12_MODELS:
        run_k12(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_12', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    Zs = {name: dict(np.load('/content/drive/MyDrive/logit-thermo/A2_12' + f'/{name}.npz')) for name, _ in K12_MODELS}
    r = analyze12(Zs)
    r.update(gate='docs/A2_12_GATE.md', a2_12_sha256='08363b2061241d94875dbf07496c6e98b991465993a768c52bf75c6691f2305a', a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', a2_7_collect_sha256='8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550',
             torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
             gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision())
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_12' + '/a2_12_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-12（1 回の禁止で乱れるトークンの数）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-12 の判定を A2_12/a2_12_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
